In [15]:
import json
import datetime

def convert_to_anner(input_file, output_file, annotator):
    STATUS = 'Candidate'
    COLOR_LIST = [
        'red-11', 'blue-11', 'light-green-11', 'yellow-11', 
        'purple-11', 'orange-11', 'teal-11', 'pink-11', 
        'brown-11', 'cyan-11', 'lime-11'
    ]
    
    with open(input_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    # Get timestamp
    timestamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

    # Build classes dynamically from input
    classes = []
    class_name_to_id = {}
    for idx, class_name in enumerate(data['classes'], start=1):
        color = COLOR_LIST[(idx - 1) % len(COLOR_LIST)]  # Rotate colors if needed
        classes.append({
            'id': idx,
            'name': class_name,
            'color': color
        })
        class_name_to_id[class_name] = class_name  # Just map to itself for annotation use

    # Build annotations
    annotations = []
    for text, ann in data['annotations']:
        entities = []
        for start, end, label in ann['entities']:
            entities.append([
                None,
                start,
                end,
                [
                    [
                        label,
                        STATUS,
                        timestamp,
                        annotator
                    ]
                ]
            ])
        annotations.append([None, text, {'entities': entities}])

    # Final structure
    output_data = {
        'classes': classes,
        'annotations': annotations
    }
    
    # Write to output file
    with open(output_file, 'w', encoding='utf-8') as file:
        json.dump(output_data, file, indent=2)

    print(f'Conversion completed. Output saved to {output_file}')


In [16]:
if '__name__' == '__main__':
    annotator = 'gpt-4o'
    input_file = 'Robles-2015_spacy.json'
    output_file = 'Robles-2015_AnNER.json'
    convert_to_anner(input_file, output_file, annotator)